# BlocksNet Urban Planning ReAct Agent

LangGraph ReAct-агент для городского анализа на основе библиотеки [blocksnet](https://github.com/aimclub/blocksnet).

Агент получает задачу на естественном языке, использует инструменты blocksnet для анализа,
сохраняет результаты в `outputs/` и возвращает структурированный ответ:
```python
{
    'input':  str,
    'output': str,
    'log':    list[BaseMessage]  # HumanMessage + AIMessage
}
```

In [2]:
# Раскомментируй и запусти один раз, если пакеты не установлены
# %pip install langgraph langchain-openai langchain-core python-dotenv geopandas -q

In [3]:
import os
import json
import pathlib
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import geopandas as gpd
from dotenv import load_dotenv

from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

# blocksnet — сетевой анализ
from blocksnet.analysis.network import (
    mean_accessibility,
    median_accessibility,
    max_accessibility,
    calculate_connectivity,
    area_accessibility,
    land_use_accessibility,
)

# blocksnet — обеспеченность сервисами
from blocksnet.analysis.provision import (
    competitive_provision,
    shared_provision,
    provision_strong_total,
    provision_weak_total,
)

# blocksnet — прочий анализ
from blocksnet.analysis.centrality import services_centrality, population_centrality
from blocksnet.analysis.diversity import shannon_diversity
from blocksnet.analysis.indicators import (
    calculate_density_indicators,
    calculate_development_indicators,
)
from blocksnet.analysis.services import services_density, services_count, services_collocation

# blocksnet.blocks.__init__ пустой — импортируем из подмодулей напрямую
from blocksnet.blocks.aggregation import aggregate_objects
from blocksnet.blocks.assignment import assign_land_use
from blocksnet.relations import generate_adjacency_graph

# blocksnet — перечисления и конфигурация
from blocksnet.enums import LandUse
from blocksnet.config import service_types_config

print("Импорты выполнены успешно.")

c:\Users\Eynor\AppData\Local\Programs\Python\Python311\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


Импорты выполнены успешно.


In [4]:
load_dotenv()  # ищет .env начиная от текущей папки и выше

# Ноутбук находится в examples/, данные — в ../data/
_HERE = pathlib.Path(__file__).parent if "__file__" in dir() else pathlib.Path.cwd()
DATA_DIR = _HERE.parent / "data"
if not DATA_DIR.exists():          # fallback: если запуск из корня проекта
    DATA_DIR = _HERE / "data"

OUTPUT_DIR = _HERE.parent / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

# Глобальный кэш загруженных данных
DATA_CACHE: dict = {}

assert os.environ.get("FP2MP_CHAT_URL"), "Не задана переменная FP2MP_CHAT_URL"
assert os.environ.get("FP2MP_API_KEY"),  "Не задана переменная FP2MP_API_KEY"
print(f"DATA_DIR  : {DATA_DIR.resolve()}")
print(f"OUTPUT_DIR: {OUTPUT_DIR.resolve()}")
print("Переменные окружения загружены.")

DATA_DIR  : P:\AI_asistent\ITMO\blocksnet-agent\data
OUTPUT_DIR: P:\AI_asistent\ITMO\blocksnet-agent\outputs
Переменные окружения загружены.


In [5]:
llm = ChatOpenAI(
    base_url=os.environ["FP2MP_CHAT_URL"],
    api_key=os.environ["FP2MP_API_KEY"],
    model=os.environ.get("FP2MP_MODEL", "gpt-4o"),
    temperature=0,
    max_tokens=4096,
)
print(f"LLM: {llm.model_name}")

LLM: gpt-4o


## Инструменты — загрузка данных

In [6]:
@tool
def load_blocks() -> str:
    """Загружает GeoDataFrame городских кварталов с данными о сервисах из файла data/blocks_with_services.gpkg.
    Необходимо вызвать перед любым аналитическим инструментом, работающим с кварталами.
    Возвращает сводку: форму, CRS, распределение землепользования, статистику населения и список типов сервисов."""
    try:
        blocks = gpd.read_file(DATA_DIR / "blocks_with_services.gpkg")

        # Приводим land_use из строки 'LandUse.RESIDENTIAL' к enum LandUse
        def _parse_land_use(val):
            if isinstance(val, LandUse):
                return val
            if isinstance(val, str):
                key = val.split(".")[-1].upper()
                try:
                    return LandUse[key]
                except KeyError:
                    return val
            return val

        blocks["land_use"] = blocks["land_use"].apply(_parse_land_use)

        # Площадь квартала в кв. м (нужна для area_accessibility и density indicators)
        blocks["site_area"] = blocks.geometry.area

        DATA_CACHE["blocks"] = blocks

        # Формируем ответ
        lu_counts = blocks["land_use"].value_counts().to_dict()
        pop_stats = blocks["population"].describe().to_dict() if "population" in blocks.columns else {}
        svc_cols = [c.replace("capacity_", "") for c in blocks.columns if c.startswith("capacity_")]

        lines = [
            f"Кварталы загружены: {blocks.shape[0]} строк, {blocks.shape[1]} столбцов.",
            f"CRS: {blocks.crs}",
            f"Землепользование (land_use): { {str(k): v for k, v in lu_counts.items()} }",
        ]
        if pop_stats:
            lines.append(
                f"Население — всего: {int(blocks['population'].sum())}, "
                f"среднее: {pop_stats.get('mean', 0):.1f}, "
                f"медиана: {pop_stats.get('50%', 0):.1f}"
            )
        lines.append(f"Типов сервисов: {len(svc_cols)}: {', '.join(svc_cols[:10])} {'...' if len(svc_cols) > 10 else ''}")
        return "\n".join(lines)
    except Exception as e:
        return f"Ошибка при загрузке кварталов: {e}"


@tool
def load_accessibility_matrix() -> str:
    """Загружает предвычисленную матрицу доступности (время в пути между кварталами) из data/acc_mx.pickle.
    Необходимо вызвать перед любым инструментом сетевого анализа или обеспеченности.
    Возвращает размерность матрицы и статистику времени в пути."""
    try:
        acc_mx = pd.read_pickle(DATA_DIR / "acc_mx.pickle")
        DATA_CACHE["acc_mx"] = acc_mx
        flat = acc_mx.values[acc_mx.values > 0]
        return (
            f"Матрица доступности загружена: {acc_mx.shape[0]}×{acc_mx.shape[1]}, dtype={acc_mx.dtypes.iloc[0]}.\n"
            f"Время в пути (мин) — мин: {flat.min():.1f}, макс: {flat.max():.1f}, "
            f"среднее: {flat.mean():.1f}, медиана: {np.median(flat):.1f}."
        )
    except Exception as e:
        return f"Ошибка при загрузке матрицы: {e}"


@tool
def list_cached_data() -> str:
    """Возвращает список всех наборов данных, загруженных в кэш, с их размерами.
    Используй перед загрузкой, чтобы не загружать повторно."""
    if not DATA_CACHE:
        return "Кэш пуст. Загрузи данные с помощью load_blocks() и load_accessibility_matrix()."
    parts = []
    for key, val in DATA_CACHE.items():
        if hasattr(val, "shape"):
            parts.append(f"{key}: {val.shape}")
        elif hasattr(val, "number_of_nodes"):
            parts.append(f"{key}: граф ({val.number_of_nodes()} узлов, {val.number_of_edges()} рёбер)")
        else:
            parts.append(f"{key}: {type(val).__name__}")
    return "Данные в кэше:\n" + "\n".join(parts)


@tool
def list_service_types() -> str:
    """Возвращает список всех доступных типов сервисов из данных кварталов.
    Используй, чтобы узнать допустимые значения аргумента service_type.
    Требует предварительного вызова load_blocks()."""
    try:
        if "blocks" not in DATA_CACHE:
            return "Ошибка: сначала вызови load_blocks()."
        svc_cols = sorted(
            c.replace("capacity_", "")
            for c in DATA_CACHE["blocks"].columns
            if c.startswith("capacity_")
        )
        return f"Доступные типы сервисов ({len(svc_cols)} шт.):" + "\n" + ", ".join(svc_cols)
    except Exception as e:
        return f"Ошибка: {e}"


@tool
def get_block_info(block_id: int) -> str:
    """Возвращает подробную информацию о конкретном квартале по его целочисленному ID.
    Показывает: землепользование, население, площадь, все мощности сервисов и сохранённые результаты анализа.
    Требует предварительного вызова load_blocks()."""
    try:
        if "blocks" not in DATA_CACHE:
            return "Ошибка: сначала вызови load_blocks()."
        blocks = DATA_CACHE["blocks"]
        row = blocks[blocks.index == block_id]
        if row.empty:
            # Попробуем по block_id как колонке
            if "block_id" in blocks.columns:
                row = blocks[blocks["block_id"] == block_id]
        if row.empty:
            return f"Квартал с ID={block_id} не найден. Допустимый диапазон: {blocks.index.min()}–{blocks.index.max()}."
        s = row.iloc[0].drop(labels=["geometry"], errors="ignore")
        lines = [f"Квартал ID={block_id}:"]
        for col, val in s.items():
            if pd.notna(val) and val != 0:
                lines.append(f"  {col}: {val}")
        return "\n".join(lines)
    except Exception as e:
        return f"Ошибка: {e}"


print("Инструменты загрузки данных определены.")

Инструменты загрузки данных определены.


## Инструменты — сетевой анализ доступности

In [7]:
def _require(*keys: str) -> str | None:
    """Возвращает строку с ошибкой, если нужные ключи отсутствуют в кэше."""
    missing = [k for k in keys if k not in DATA_CACHE]
    if missing:
        loaders = {"blocks": "load_blocks()", "acc_mx": "load_accessibility_matrix()",
                   "adjacency_graph": "build_adjacency_graph()"}
        hints = ", ".join(loaders.get(k, f"load_{k}()") for k in missing)
        return f"Ошибка: отсутствуют данные {missing}. Сначала вызови: {hints}."
    return None


def _acc_summary(df: pd.DataFrame, col: str = "accessibility") -> str:
    vals = df[col].dropna()
    top5 = df.nsmallest(5, col)[[col]]
    bot5 = df.nlargest(5, col)[[col]]
    return (
        f"Мин: {vals.min():.2f}, макс: {vals.max():.2f}, "
        f"среднее: {vals.mean():.2f}, медиана: {vals.median():.2f}.\n"
        f"Топ-5 наиболее доступных (наименьшее время):\n{top5.to_string()}\n"
        f"Топ-5 наименее доступных (наибольшее время):\n{bot5.to_string()}"
    )


@tool
def compute_mean_accessibility(out: bool = True) -> str:
    """Вычисляет среднее время доступности для каждого квартала по матрице доступности.
    out=True — исходящая доступность (как доступны другие кварталы ИЗ данного).
    out=False — входящая доступность (насколько данный квартал доступен ИЗ других).
    Сохраняет результат в outputs/mean_accessibility.csv.
    Требует load_blocks() и load_accessibility_matrix()."""
    try:
        err = _require("acc_mx")
        if err: return err
        df = mean_accessibility(DATA_CACHE["acc_mx"], out=out)
        df.to_csv(OUTPUT_DIR / "mean_accessibility.csv")
        DATA_CACHE["mean_accessibility"] = df
        return f"Средняя доступность (out={out}) вычислена.\n" + _acc_summary(df)
    except Exception as e:
        return f"Ошибка: {e}"


@tool
def compute_median_accessibility(out: bool = True) -> str:
    """Вычисляет медианное время доступности для каждого квартала.
    Менее чувствительна к выбросам, чем среднее.
    Сохраняет результат в outputs/median_accessibility.csv.
    Требует load_accessibility_matrix()."""
    try:
        err = _require("acc_mx")
        if err: return err
        df = median_accessibility(DATA_CACHE["acc_mx"], out=out)
        df.to_csv(OUTPUT_DIR / "median_accessibility.csv")
        DATA_CACHE["median_accessibility"] = df
        return f"Медианная доступность (out={out}) вычислена.\n" + _acc_summary(df)
    except Exception as e:
        return f"Ошибка: {e}"


@tool
def compute_max_accessibility(out: bool = True) -> str:
    """Вычисляет максимальное время доступности для каждого квартала.
    Показывает наихудший сценарий доступности.
    Сохраняет результат в outputs/max_accessibility.csv.
    Требует load_accessibility_matrix()."""
    try:
        err = _require("acc_mx")
        if err: return err
        df = max_accessibility(DATA_CACHE["acc_mx"], out=out)
        df.to_csv(OUTPUT_DIR / "max_accessibility.csv")
        DATA_CACHE["max_accessibility"] = df
        return f"Максимальная доступность (out={out}) вычислена.\n" + _acc_summary(df)
    except Exception as e:
        return f"Ошибка: {e}"


@tool
def compute_connectivity(accessibility_key: str = "mean_accessibility") -> str:
    """Вычисляет связность транспортной сети — обратную величину к доступности (1 / время).
    Высокая связность = хорошее транспортное соединение с остальными кварталами.
    accessibility_key: ключ в кэше с уже вычисленной доступностью (например, 'mean_accessibility').
    Требует предварительного вычисления доступности."""
    try:
        if accessibility_key not in DATA_CACHE:
            return f"Ошибка: '{accessibility_key}' не в кэше. Сначала вычисли доступность."
        df = calculate_connectivity(DATA_CACHE[accessibility_key])
        df.to_csv(OUTPUT_DIR / "connectivity.csv")
        DATA_CACHE["connectivity"] = df
        col = df.columns[0]
        vals = df[col].dropna()
        top5 = df.nlargest(5, col)[[col]]
        return (
            f"Связность вычислена.\nМин: {vals.min():.4f}, макс: {vals.max():.4f}, "
            f"среднее: {vals.mean():.4f}.\nТоп-5 наиболее связных кварталов:\n{top5.to_string()}"
        )
    except Exception as e:
        return f"Ошибка: {e}"


@tool
def compute_land_use_accessibility(land_use: str, out: bool = True) -> str:
    """Вычисляет доступность до кварталов определённого типа землепользования.
    land_use: одно из RESIDENTIAL, BUSINESS, RECREATION, INDUSTRIAL, TRANSPORT, SPECIAL, AGRICULTURE.
    out=True — для каждого квартала среднее время до кварталов заданного типа.
    Требует load_blocks() и load_accessibility_matrix()."""
    try:
        err = _require("blocks", "acc_mx")
        if err: return err
        lu = LandUse[land_use.upper()]
        df = land_use_accessibility(DATA_CACHE["acc_mx"], DATA_CACHE["blocks"], land_use=lu, out=out)
        df.to_csv(OUTPUT_DIR / f"lu_accessibility_{land_use.lower()}.csv")
        DATA_CACHE[f"lu_accessibility_{land_use.lower()}"] = df
        return f"Доступность до зон {land_use} вычислена.\n" + _acc_summary(df)
    except KeyError:
        valid = [e.name for e in LandUse]
        return f"Неверный тип: '{land_use}'. Допустимые: {valid}"
    except Exception as e:
        return f"Ошибка: {e}"


@tool
def compute_area_accessibility(out: bool = True) -> str:
    """Вычисляет площадно-взвешенную доступность: учитывает площадь целевых кварталов.
    Большие кварталы вносят больший вес в среднее время в пути.
    Требует load_blocks() и load_accessibility_matrix()."""
    try:
        err = _require("blocks", "acc_mx")
        if err: return err
        df = area_accessibility(DATA_CACHE["acc_mx"], DATA_CACHE["blocks"], out=out)
        df.to_csv(OUTPUT_DIR / "area_accessibility.csv")
        DATA_CACHE["area_accessibility"] = df
        return f"Площадно-взвешенная доступность (out={out}) вычислена.\n" + _acc_summary(df)
    except Exception as e:
        return f"Ошибка: {e}"


print("Инструменты сетевого анализа определены.")

Инструменты сетевого анализа определены.


## Инструменты — обеспеченность сервисами

In [8]:
@tool
def compute_service_provision(
    service_type: str,
    accessibility_minutes: int = 15,
    max_depth: int = 1,
) -> str:
    """Вычисляет конкурентную обеспеченность населения конкретным типом сервиса.
    service_type: тип сервиса (например 'school', 'kindergarten', 'pharmacy').
      Используй list_service_types() для получения допустимых значений.
    accessibility_minutes: порог транспортной доступности (например 15 для школ, 60 для больниц).
    max_depth: число итераций перераспределения (1=быстро, 3=детальнее, но медленнее).
    Сохраняет results в outputs/provision_{service_type}.csv.
    Требует load_blocks() и load_accessibility_matrix()."""
    try:
        err = _require("blocks", "acc_mx")
        if err: return err

        cap_col = f"capacity_{service_type}"
        if cap_col not in DATA_CACHE["blocks"].columns:
            return f"Ошибка: тип сервиса '{service_type}' не найден. Вызови list_service_types()."

        service_df = DATA_CACHE["blocks"][["population", cap_col]].copy()
        service_df = service_df.rename(columns={cap_col: "capacity"})
        service_df["capacity"] = service_df["capacity"].fillna(0).astype(int)
        service_df["population"] = service_df["population"].fillna(0).astype(int)

        blocks_prov, links_df = competitive_provision(
            service_df,
            DATA_CACHE["acc_mx"],
            accessibility_minutes,
            max_depth=max_depth,
        )

        blocks_prov.to_csv(OUTPUT_DIR / f"provision_{service_type}.csv")
        if links_df is not None:
            links_df.to_csv(OUTPUT_DIR / f"links_{service_type}.csv")
        DATA_CACHE[f"provision_{service_type}"] = blocks_prov

        strong = provision_strong_total(blocks_prov)
        weak   = provision_weak_total(blocks_prov)

        # Блоки без обеспеченности
        if "provision_strong" in blocks_prov.columns:
            fully    = (blocks_prov["provision_strong"] >= 1.0).sum()
            partial  = ((blocks_prov["provision_strong"] > 0) & (blocks_prov["provision_strong"] < 1.0)).sum()
            none_cnt = (blocks_prov["provision_strong"] == 0).sum()
            detail = (
                f"Полная обеспеченность: {fully} кварталов, "
                f"частичная: {partial}, отсутствует: {none_cnt}."
            )
        else:
            detail = ""

        return (
            f"Обеспеченность сервисом '{service_type}' (порог {accessibility_minutes} мин):\n"
            f"  Суммарная сильная обеспеченность: {strong:.3f}\n"
            f"  Суммарная слабая обеспеченность:  {weak:.3f}\n"
            f"  {detail}\n"
            f"Сохранено в outputs/provision_{service_type}.csv"
        )
    except Exception as e:
        return f"Ошибка: {e}"


@tool
def compute_shared_provision(
    service_type: str,
    accessibility_minutes: int = 15,
) -> str:
    """Вычисляет совместную (населённую) обеспеченность сервисом.
    Показывает, какая доля населения каждого квартала имеет доступ к сервису
    в пределах заданного порога доступности.
    Требует load_blocks() и load_accessibility_matrix()."""
    try:
        err = _require("blocks", "acc_mx")
        if err: return err

        cap_col = f"capacity_{service_type}"
        if cap_col not in DATA_CACHE["blocks"].columns:
            return f"Ошибка: тип сервиса '{service_type}' не найден."

        service_df = DATA_CACHE["blocks"][["population", cap_col]].copy()
        service_df = service_df.rename(columns={cap_col: "capacity"})
        service_df["capacity"] = service_df["capacity"].fillna(0).astype(int)
        service_df["population"] = service_df["population"].fillna(0).astype(int)

        result_df = shared_provision(service_df, DATA_CACHE["acc_mx"], accessibility_minutes)
        result_df.to_csv(OUTPUT_DIR / f"shared_provision_{service_type}.csv")
        DATA_CACHE[f"shared_provision_{service_type}"] = result_df

        col = [c for c in result_df.columns if "provision" in c.lower()]
        summary = result_df[col].describe().to_string() if col else result_df.describe().to_string()
        return (
            f"Совместная обеспеченность '{service_type}' (порог {accessibility_minutes} мин):\n"
            f"{summary}"
        )
    except Exception as e:
        return f"Ошибка: {e}"


print("Инструменты обеспеченности сервисами определены.")

Инструменты обеспеченности сервисами определены.


## Инструменты — сервисы, централь, разнообразие

In [9]:
@tool
def compute_services_density() -> str:
    """Вычисляет плотность сервисов (число объектов на кв. км) для каждого квартала.
    Требует load_blocks()."""
    try:
        err = _require("blocks")
        if err: return err
        df = services_density(DATA_CACHE["blocks"])
        df.to_csv(OUTPUT_DIR / "services_density.csv")
        DATA_CACHE["services_density"] = df
        return f"Плотность сервисов вычислена.\n{df.describe().to_string()}"
    except Exception as e:
        return f"Ошибка: {e}"


@tool
def compute_services_count() -> str:
    """Подсчитывает количество объектов каждого типа сервиса по кварталам.
    Требует load_blocks()."""
    try:
        err = _require("blocks")
        if err: return err
        df = services_count(DATA_CACHE["blocks"])
        df.to_csv(OUTPUT_DIR / "services_count.csv")
        DATA_CACHE["services_count"] = df
        return f"Количество сервисов вычислено.\n{df.describe().to_string()}"
    except Exception as e:
        return f"Ошибка: {e}"


@tool
def compute_services_collocation() -> str:
    """Анализирует совместное расположение (колокацию) типов сервисов в кварталах.
    Выявляет, какие сервисы часто встречаются вместе.
    Требует load_blocks()."""
    try:
        err = _require("blocks")
        if err: return err
        df = services_collocation(DATA_CACHE["blocks"])
        df.to_csv(OUTPUT_DIR / "services_collocation.csv")
        DATA_CACHE["services_collocation"] = df
        return f"Колокация сервисов вычислена.\n{df.to_string()[:1000]}"
    except Exception as e:
        return f"Ошибка: {e}"


@tool
def compute_shannon_diversity() -> str:
    """Вычисляет индекс разнообразия Шеннона для распределения сервисов по кварталам.
    Высокое значение = более разнообразный набор сервисов.
    Сохраняет результат в outputs/shannon_diversity.csv.
    Требует load_blocks()."""
    try:
        err = _require("blocks")
        if err: return err
        df = shannon_diversity(DATA_CACHE["blocks"])
        df.to_csv(OUTPUT_DIR / "shannon_diversity.csv")
        DATA_CACHE["shannon_diversity"] = df
        col = df.columns[0]
        top5 = df.nlargest(5, col)[[col]]
        bot5 = df.nsmallest(5, col)[[col]]
        vals = df[col].dropna()
        return (
            f"Индекс Шеннона вычислен.\n"
            f"Мин: {vals.min():.4f}, макс: {vals.max():.4f}, среднее: {vals.mean():.4f}.\n"
            f"Топ-5 кварталов по разнообразию сервисов:\n{top5.to_string()}\n"
            f"5 кварталов с наименьшим разнообразием:\n{bot5.to_string()}"
        )
    except Exception as e:
        return f"Ошибка: {e}"


@tool
def compute_services_centrality() -> str:
    """Вычисляет составной индекс центральности каждого квартала на основе:
    - транспортной связности (connectivity)
    - разнообразия сервисов (diversity)
    - плотности сервисов (density)
    Сохраняет результат в outputs/services_centrality.csv.
    Требует load_blocks() и load_accessibility_matrix()."""
    try:
        err = _require("blocks", "acc_mx")
        if err: return err
        # Правильный порядок аргументов: (accessibility_matrix, blocks_df)
        df = services_centrality(DATA_CACHE["acc_mx"], DATA_CACHE["blocks"])
        df.to_csv(OUTPUT_DIR / "services_centrality.csv")
        DATA_CACHE["services_centrality"] = df
        col = df.columns[0]
        top10 = df.nlargest(10, col)[[col]]
        return (
            f"Централность сервисов вычислена.\n"
            f"Топ-10 наиболее центральных кварталов:\n{top10.to_string()}"
        )
    except Exception as e:
        return f"Ошибка: {e}"


@tool
def compute_population_centrality() -> str:
    """Вычисляет центральность кварталов на основе численности населения и смежности.
    Требует load_blocks() и build_adjacency_graph()."""
    try:
        err = _require("blocks", "adjacency_graph")
        if err: return err
        df = population_centrality(DATA_CACHE["blocks"], DATA_CACHE["adjacency_graph"])
        df.to_csv(OUTPUT_DIR / "population_centrality.csv")
        DATA_CACHE["population_centrality"] = df
        col = df.columns[0]
        top10 = df.nlargest(10, col)[[col]]
        return f"Центральность по населению вычислена.\nТоп-10:\n{top10.to_string()}"
    except Exception as e:
        return f"Ошибка: {e}"


print("Инструменты сервисов и централности определены.")

Инструменты сервисов и централности определены.


## Инструменты — индикаторы и граф смежности

In [10]:
@tool
def compute_density_indicators() -> str:
    """Вычисляет индикаторы плотности городской застройки для каждого квартала:
    FSI (Floor Space Index), GSI (Ground Space Index), L (число этажей), OSR, MXI.
    Необходимы для морфотипологической классификации.
    Сохраняет результат в outputs/density_indicators.csv.
    Требует load_blocks()."""
    try:
        err = _require("blocks")
        if err: return err
        df = calculate_density_indicators(DATA_CACHE["blocks"])
        df.to_csv(OUTPUT_DIR / "density_indicators.csv")
        DATA_CACHE["density_indicators"] = df
        return f"Индикаторы плотности вычислены.\n{df.describe().to_string()}"
    except Exception as e:
        return f"Ошибка: {e}"


@tool
def compute_development_indicators() -> str:
    """Вычисляет индикаторы освоенности территории (development indicators):
    нормы жилой площади, производственных площадей, нагрузки на инфраструктуру.
    Сохраняет в outputs/development_indicators.csv.
    Требует load_blocks()."""
    try:
        err = _require("blocks")
        if err: return err
        df = calculate_development_indicators(DATA_CACHE["blocks"])
        df.to_csv(OUTPUT_DIR / "development_indicators.csv")
        DATA_CACHE["development_indicators"] = df
        return f"Индикаторы освоенности вычислены.\n{df.describe().to_string()}"
    except Exception as e:
        return f"Ошибка: {e}"


@tool
def build_adjacency_graph(buffer_size: int = 0) -> str:
    """Строит граф пространственной смежности городских кварталов.
    buffer_size=0 — соединяет только кварталы с общей границей.
    Больший buffer_size соединяет и близко расположенные кварталы.
    ПРЕДУПРЕЖДЕНИЕ: занимает 20–60 секунд для 3113 кварталов.
    Требует load_blocks()."""
    try:
        err = _require("blocks")
        if err: return err
        G = generate_adjacency_graph(DATA_CACHE["blocks"], buffer_size=buffer_size)
        DATA_CACHE["adjacency_graph"] = G
        degrees = [d for _, d in G.degree()]
        avg_deg = np.mean(degrees) if degrees else 0
        return (
            f"Граф смежности построен: {G.number_of_nodes()} узлов, {G.number_of_edges()} рёбер.\n"
            f"Средняя степень узла: {avg_deg:.2f}."
        )
    except Exception as e:
        return f"Ошибка: {e}"


@tool
def get_analysis_results(result_key: str) -> str:
    """Извлекает из кэша краткую сводку ранее вычисленного результата.
    result_key: ключ в кэше (например 'mean_accessibility', 'provision_school', 'shannon_diversity').
    Используй list_cached_data() для просмотра доступных ключей."""
    try:
        if result_key not in DATA_CACHE:
            available = list(DATA_CACHE.keys())
            return f"Ключ '{result_key}' не найден. Доступные: {available}"
        val = DATA_CACHE[result_key]
        if isinstance(val, (pd.DataFrame, gpd.GeoDataFrame)):
            return (
                f"Результат '{result_key}' ({val.shape}):\n"
                f"Статистика:\n{val.describe().to_string()}\n\n"
                f"Первые строки:\n{val.head(5).to_string()}"
            )
        return f"'{result_key}': {str(val)[:500]}"
    except Exception as e:
        return f"Ошибка: {e}"


print("Инструменты индикаторов и графа определены.")

Инструменты индикаторов и графа определены.


## Сборка ReAct-агента

In [11]:
SYSTEM_PROMPT = """Ты — аналитик городского планирования, работающий с библиотекой BlocksNet.
У тебя есть инструменты для анализа городских кварталов российского города.

ПРАВИЛА РАБОТЫ:
1. Перед любым анализом вызови load_blocks() и load_accessibility_matrix().
2. Используй list_cached_data() чтобы не загружать данные повторно.
3. Для получения допустимых типов сервисов используй list_service_types().
4. Возвращай количественные результаты с интерпретацией.
5. При ошибке в инструменте не прерывай работу — объясни проблему и продолжи.
6. Все результаты сохраняются в папку outputs/.
"""

all_tools = [
    # Загрузка данных
    load_blocks,
    load_accessibility_matrix,
    list_cached_data,
    list_service_types,
    get_block_info,
    get_analysis_results,
    # Сетевой анализ
    compute_mean_accessibility,
    compute_median_accessibility,
    compute_max_accessibility,
    compute_connectivity,
    compute_land_use_accessibility,
    compute_area_accessibility,
    # Обеспеченность
    compute_service_provision,
    compute_shared_provision,
    # Сервисы и централность
    compute_services_density,
    compute_services_count,
    compute_services_collocation,
    compute_shannon_diversity,
    compute_services_centrality,
    compute_population_centrality,
    # Индикаторы и граф
    compute_density_indicators,
    compute_development_indicators,
    build_adjacency_graph,
]

agent = create_react_agent(
    model=llm,
    tools=all_tools,
    prompt=SYSTEM_PROMPT,
)

print(f"Агент создан с {len(all_tools)} инструментами.")

Агент создан с 23 инструментами.


In [12]:
def run_agent(task: str) -> dict:
    """Запускает агента на задаче и возвращает структурированный результат.

    Returns
    -------
    dict с ключами:
        input  : str            — исходная задача
        output : str            — финальный ответ агента
        log    : list[BaseMessage]  — вся переписка (HumanMessage + AIMessage)
    """
    result = agent.invoke({"messages": [HumanMessage(content=task)]})
    messages: list[BaseMessage] = result["messages"]

    # Оставляем только HumanMessage и AIMessage (без ToolMessage)
    log = [m for m in messages if isinstance(m, (HumanMessage, AIMessage))]

    # Финальный ответ — последний AIMessage с непустым контентом
    final_output = next(
        (m.content for m in reversed(messages) if isinstance(m, AIMessage) and m.content),
        "Ответ не получен.",
    )

    return {"input": task, "output": final_output, "log": log}


def print_result(result: dict, max_log_chars: int = 400) -> None:
    """Красиво печатает результат run_agent."""
    print("=" * 70)
    print(f"INPUT:\n  {result['input']}")
    print(f"\nOUTPUT:\n{result['output']}")
    print(f"\nLOG ({len(result['log'])} сообщений):")
    for msg in result["log"]:
        role = "USER " if isinstance(msg, HumanMessage) else "AGENT"
        content = str(msg.content)
        if len(content) > max_log_chars:
            content = content[:max_log_chars] + " ...[обрезано]"
        print(f"  [{role}] {content}")
    print("=" * 70)


print("Функции run_agent и print_result определены.")

Функции run_agent и print_result определены.


---

## Пример 1 — Анализ сетевой доступности

Агент вычисляет среднюю транспортную доступность и связность кварталов,
выявляет наиболее и наименее доступные районы.

In [ ]:
result1 = run_agent(
    "Проанализируй транспортную доступность городских кварталов. "
    "Вычисли среднюю доступность и связность. "
    "Какие кварталы наиболее и наименее доступны?"
)
print_result(result1)

## Пример 2 — Обеспеченность школами

Агент оценивает конкурентную обеспеченность населения школами
с порогом транспортной доступности 15 минут.

In [13]:
result2 = run_agent(
    "Оцени обеспеченность населения школами. "
    "Используй порог транспортной доступности 15 минут. "
    "Какие кварталы испытывают наибольший дефицит школьных мест?"
)
print_result(result2)

APIConnectionError: Connection error.

## Пример 3 — Разнообразие сервисов и централность

Агент вычисляет индекс разнообразия Шеннона и централность кварталов,
выявляет зоны с наибольшим и наименьшим разнообразием сервисов.

In [ ]:
result3 = run_agent(
    "Вычисли индекс разнообразия Шеннона для сервисов по кварталам "
    "и централность кварталов на основе сервисов. "
    "Какие кварталы имеют наиболее и наименее разнообразный набор услуг?"
)
print_result(result3)

---

## Просмотр сохранённых результатов

In [ ]:
from IPython.display import display

csv_files = sorted(OUTPUT_DIR.glob("*.csv"))
print(f"Сохранено файлов в outputs/: {len(csv_files)}")
for f in csv_files:
    df = pd.read_csv(f)
    print(f"\n=== {f.name} ({df.shape}) ===")
    display(df.head(3))

In [ ]:
# Проверяем типы объектов в log
for result in [result1, result2, result3]:
    types = [type(m).__name__ for m in result["log"]]
    print(f"Task: '{result['input'][:50]}...'")
    print(f"  log types: {types}")
    print(f"  все BaseMessage: {all(isinstance(m, BaseMessage) for m in result['log'])}")

---

## Примечания о производительности

| Инструмент | Время выполнения |
|---|---|
| `load_blocks()` | 2–5 с |
| `load_accessibility_matrix()` | 1–3 с |
| `compute_*_accessibility()` | 2–10 с |
| `compute_shannon_diversity()` | 1–3 с |
| `compute_service_provision()` | 15–60 с |
| `compute_services_centrality()` | 5–15 с |
| `build_adjacency_graph()` | 20–60 с |
| `compute_density_indicators()` | 1–5 с |

> **Важно:** матрица доступности уже предвычислена и загружается из pickle-файла.
> Не вызывай `calculate_accessibility_matrix()` напрямую — это займёт несколько часов.